In [5]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
import xgboost as xgb
from sklearn.ensemble import VotingClassifier

In [2]:
train_df = pd.read_csv('/kaggle/input/datasets/shauryasayshi/smartphone-addiction-dataset/train (1).csv')
train_df.info()

train_df.isna().sum()
test_df = pd.read_csv('/kaggle/input/datasets/shauryasayshi/smartphone-addiction-dataset/test (1).csv')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       691369 non-null  int64  
 1   age                      662440 non-null  float64
 2   daily_screen_time_hours  595515 non-null  float64
 3   social_media_hours       557374 non-null  float64
 4   gaming_hours             564548 non-null  float64
 5   work_study_hours         639851 non-null  float64
 6   sleep_hours              646889 non-null  float64
 7   notifications_per_day    623785 non-null  float64
 8   app_opens_per_day        610659 non-null  float64
 9   weekend_screen_time      579306 non-null  float64
 10  gender                   662335 non-null  object 
 11  stress_level             636221 non-null  object 
 12  academic_work_impact     647145 non-null  object 
 13  addicted_label           691369 non-null  int64  
dtypes: f

In [7]:
X =  train_df.drop(columns=["addicted_label", "id"])
y = train_df["addicted_label"]


X_test = test_df.copy()


categorical_columns = ["stress_level","gender","academic_work_impact"]
numeric_cols = [c for c in X.columns if c not in categorical_columns]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler())
        ]), numeric_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(drop="first"))
        ]), categorical_columns)
    ]
)


hgb_model = HistGradientBoostingClassifier(
    min_samples_leaf=100,
    max_leaf_nodes=31,
    max_iter=300,
    max_depth=None,
    learning_rate=0.1,
    random_state=42
)

pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", hgb_model)
])


pipe.fit(X, y)

X_test['addicted_label'] = pipe.predict(X_test)
# print(accuracy_score(y_test, y_pred))
X_test = X_test[["id", "addicted_label"]]

# X_test.to_csv("smartphone_addiction_comp_submission.csv", index=False)

In [8]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__max_iter": [100, 200, 300],
    "model__max_leaf_nodes": [15, 31, 63],
    "model__min_samples_leaf": [20, 50, 100],
    "model__max_depth": [None, 5, 10]
}

random_search = RandomizedSearchCV(
    pipe,
    param_distributions=param_dist,
    n_iter=20,          
    cv=5,
    scoring="accuracy", 
    n_jobs=-1,
    random_state=42,
    verbose=2
)

random_search.fit(X, y)

print(random_search.best_params_)
print(random_search.best_score_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
{'model__min_samples_leaf': 100, 'model__max_leaf_nodes': 31, 'model__max_iter': 300, 'model__max_depth': None, 'model__learning_rate': 0.1}
0.8972082919920202
[CV] END model__learning_rate=0.03, model__max_depth=5, model__max_iter=300, model__max_leaf_nodes=63, model__min_samples_leaf=20; total time=  44.3s
[CV] END model__learning_rate=0.03, model__max_depth=5, model__max_iter=100, model__max_leaf_nodes=15, model__min_samples_leaf=20; total time=  16.6s
[CV] END model__learning_rate=0.03, model__max_depth=5, model__max_iter=100, model__max_leaf_nodes=15, model__min_samples_leaf=20; total time=  16.6s
[CV] END model__learning_rate=0.03, model__max_depth=10, model__max_iter=100, model__max_leaf_nodes=15, model__min_samples_leaf=100; total time=  17.6s
[CV] END model__learning_rate=0.01, model__max_depth=None, model__max_iter=200, model__max_leaf_nodes=15, model__min_samples_leaf=20; total time=  32.0s
[CV] END model__learnin